# Hippocampus -> Amygdala  HFA Computation

Upstream pipeline only: load NWB -> preprocess (demean/detrend, no re-referencing) -> HFA via Hilbert envelope -> epoch around encoding1 / maintenance / probe -> pickle one file per session into `outputs/hipp_amyg_HFA_CSA_v2/hfa_cache/`.

**Re-run this notebook when:** preprocessing flags change, HFA band/sub-band config changes, the bad-channel column list changes, or the epoch windows change.

Downstream FA + RRR analyses live in `hipp_amyg_analyze.ipynb`, which loads the pickles this notebook produces.


## 1. Setup & Config

In [1]:
# Pin each worker's BLAS to a single thread BEFORE numpy/scipy/sklearn are imported.
# Without this, joblib spawns N processes and each one fans out its own BLAS pool,
# producing massive oversubscription on many-core machines.
import os
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'BLIS_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_v, '1')

import warnings, time, pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats as scistats
from scipy.signal import butter, filtfilt, hilbert, detrend

from pynwb import NWBHDF5IO

from nwb_analysis.data_loading import get_subject_files, load_nwb_file, load_lfp_safe
from nwb_analysis.regions import extract_region_channels
from nwb_analysis.config import BRAIN_REGIONS

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='pynwb')


In [2]:
PROJECT_ROOT = Path('.').resolve()
DATA_DIR = PROJECT_ROOT / '000673'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'hipp_amyg_HFA_CSA_v2'
HFA_CACHE_DIR = OUT_DIR / 'hfa_cache'
OUT_DIR.mkdir(parents=True, exist_ok=True)
HFA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Quick proof-of-concept toggle. True = fast/noisy (subsample trials,
# fewer folds, more aggressive block-averaging). False = strict Binish-
# faithful settings (all trials, 10 folds, aggregate_factor=10).
POC_MODE = True

CONFIG = {
    'data_dir': DATA_DIR,
    'out_dir': OUT_DIR,
    'hfa_cache_dir': HFA_CACHE_DIR,
    'random_seed': 20260508,

    # --- Region pair (swap via flip_direction) ---
    'source_region': 'Hippocampus',
    'target_region': 'Amygdala',
    'flip_direction': False,

    # --- Coverage filter ---
    'min_chans_per_region': 8,

    # --- Bad channel handling ---
    # Use dataset-provided flags only (no MAD-based heuristic). Checked column names
    # in order; first matching boolean column flags channels for exclusion.
    'bad_channel_flag_columns': ['bad', 'is_bad', 'epileptic', 'is_epileptic',
                                  'is_seizure_onset', 'seizure_onset',
                                  'is_resected', 'reject', 'rejected'],

    # --- Local referencing ---
    # Preprocessing — all three OFF by default (raw HFA from raw LFP).
    # Flip any flag to True to enable that step.
    'apply_demean':       True,   # subtract per-channel time mean
    'apply_detrend':      True,   # scipy.signal.detrend (linear)

    # --- HFA: multi-band ---
    'bands': {
        'theta':      {'low': 3,   'high': 7,   'n_subbands': 1},
        'alpha':      {'low': 8,   'high': 12,  'n_subbands': 1},
        'beta':       {'low': 13,  'high': 30,  'n_subbands': 1},
        'low_gamma':  {'low': 30,  'high': 55,  'n_subbands': 3},
        'high_gamma': {'low': 70,  'high': 140, 'n_subbands': 7},
    },
    'hfa_filter_order':              4,
    'hfa_baseline_window':           (-1.0, 0.0),
    'hfa_baseline_lock_event':       'encoding1',
    'hfa_baseline_bootstrap_iters':  1,

    # --- Epoching: build tensors for these three events per session ---
    'epoch_windows': {
        'encoding1':   (0.0, 2.0),
        'maintenance': (0.0, 2.5),
        'probe':       (0.0, 2.0),
    },

    # --- Trial filtering & grouping ---
    'use_only_correct': True,
    'trial_grouping': 'category',
    'category_keep': (1, 2, 3, 4, 5),

    # --- Block aggregation (used in FA + RRR) ---
    'aggregate_factor': 10,
    'aggregate_method': 'block_avg',

    # --- Section 6: Factor Analysis (Binish/Semedo-style) ---
    'fa_n_folds': 10,                   # matches Binish's CrossValFa default
    'fa_threshold': 0.95,               # smallest q with cumulative shared variance >= 0.95
    'fa_n_trials_per_session': None,    # None = use all kept trials
    # Outer (sessions) × inner (folds) <= n_cores. Defaults assume a many-core box:
    # session_n_jobs=-1 uses every core, fold_n_jobs=1 keeps the inner serial.
    'fa_session_n_jobs': 24,    # physical cores; do NOT use -1 (counts hyperthreads)
    'fa_fold_n_jobs': 1,

    # --- Section 7: RRR ---
    'rrr_n_chans': 10,
    'rrr_n_iters': 100,                 # bump to 1000 for paper figures
    'rrr_n_folds': 10,
    'rrr_alpha_grid': np.logspace(-3, 3, 13),

    # --- Threading (used by HFA sweep — I/O bound, threads are fine) ---
    'n_workers': 4,
}

# Apply direction flip
if CONFIG['flip_direction']:
    CONFIG['source_region'], CONFIG['target_region'] = CONFIG['target_region'], CONFIG['source_region']



# --- PoC override: shrink sample sizes for fast iteration ---
if POC_MODE:
    CONFIG['fa_n_trials_per_session'] = 30
    CONFIG['fa_n_folds'] = 5
    CONFIG['aggregate_factor'] = 20
    print('[POC_MODE override applied] '
          f"trials/session={CONFIG['fa_n_trials_per_session']}, "
          f"n_folds={CONFIG['fa_n_folds']}, "
          f"aggregate_factor={CONFIG['aggregate_factor']}")

print(f"Source -> Target: {CONFIG['source_region']} -> {CONFIG['target_region']}")
print('Bands:')
for _name, _spec in CONFIG['bands'].items():
    print(f"  {_name:>12s}: {_spec['low']:>3}-{_spec['high']:>3} Hz, "
          f"{_spec['n_subbands']} sub-band(s)")
print(f"Cache dir: {HFA_CACHE_DIR}")
print(f"FA parallelism: session_n_jobs={CONFIG['fa_session_n_jobs']}, fold_n_jobs={CONFIG['fa_fold_n_jobs']}")
print(f"HFA workers: {CONFIG['n_workers']}")

[POC_MODE override applied] trials/session=30, n_folds=5, aggregate_factor=20
Source -> Target: Hippocampus -> Amygdala
Bands:
         theta:   3-  7 Hz, 1 sub-band(s)
         alpha:   8- 12 Hz, 1 sub-band(s)
          beta:  13- 30 Hz, 1 sub-band(s)
     low_gamma:  30- 55 Hz, 3 sub-band(s)
    high_gamma:  70-140 Hz, 7 sub-band(s)
Cache dir: E:\SBCAT\outputs\hipp_amyg_HFA_CSA_v2\hfa_cache
FA parallelism: session_n_jobs=24, fold_n_jobs=1
HFA workers: 4


## 2. Helpers

In [3]:
# --- Preprocessing helpers ---
def load_lfp_oriented(lfp_series, n_electrodes):
    """Return LFP as (timepoints, channels), auto-correcting transposition."""
    raw = np.asarray(lfp_series.data)
    if raw.ndim == 1:
        return raw.reshape(-1, 1)
    s0, s1 = raw.shape
    if s0 == n_electrodes and s1 != n_electrodes:
        print(f'  [transpose] LFP {raw.shape} (chan, time) -> (time, chan)')
        return raw.T
    return raw


def detect_bad_channels_from_metadata(edf, candidate_columns):
    """Inspect electrodes table for any of the candidate boolean flag columns.
    Returns list of channel indices (positional) that the dataset flags as bad."""
    bad_idx = set()
    used_cols = []
    for name in candidate_columns:
        if name not in edf.columns:
            continue
        col = edf[name]
        try:
            mask = col.astype(bool)
        except Exception:
            continue
        if mask.any():
            idx = np.where(mask.values)[0]
            bad_idx.update(idx.tolist())
            used_cols.append((name, len(idx)))
    return sorted(bad_idx), used_cols


def preprocess_session_lfp(nwb_data, cfg, verbose=True):
    """Load LFP and (optionally) demean / detrend. No re-referencing."""
    lfp_series = nwb_data['lfp']['series']
    fs = nwb_data['lfp']['sampling_rate']
    edf = nwb_data['electrodes']
    n_electrodes = len(edf)

    lfp = load_lfp_oriented(lfp_series, n_electrodes).astype(np.float32)
    n_chan_lfp = lfp.shape[1]
    if verbose:
        print(f'  loaded LFP: shape={lfp.shape}, fs={fs}')

    if cfg.get('apply_demean', False):
        lfp = lfp - lfp.mean(axis=0, keepdims=True)
        if verbose: print('  [demean] applied')
    else:
        if verbose: print('  [demean] skipped')

    if cfg.get('apply_detrend', False):
        lfp = detrend(lfp, axis=0, type='linear')
        if verbose: print('  [detrend] applied')
    else:
        if verbose: print('  [detrend] skipped')

    bad_idx, used_cols = detect_bad_channels_from_metadata(edf, cfg['bad_channel_flag_columns'])
    if verbose:
        if used_cols:
            for name, n in used_cols:
                print(f'  [bad-flag] electrodes column {name!r}: {n} channels flagged')
            print(f'  union of dataset-flagged bad channels: {bad_idx}')
        else:
            print('  no dataset-flagged bad channels found in electrodes table')
    return lfp, fs, edf, {'n_chan_lfp': n_chan_lfp,
                          'bad_idx': bad_idx, 'flag_cols': used_cols}


print('preprocessing helpers defined')


preprocessing helpers defined


In [4]:
# --- HFA + epoching helpers ---
def design_subband_filter(fs, low, high, order):
    nyq = fs / 2
    return butter(order, [low / nyq, high / nyq], btype='band')


def subband_envelopes(lfp_chan, fs, edges, order):
    out = np.empty((len(edges), lfp_chan.shape[0]), dtype=np.float32)
    for i, (lo, hi) in enumerate(edges):
        b, a = design_subband_filter(fs, lo, hi, order)
        bp = filtfilt(b, a, lfp_chan)
        out[i] = np.abs(hilbert(bp)).astype(np.float32)
    return out


def bootstrap_baseline_zscore(envelope, baseline_idx, n_iters, rng):
    """Vectorized: draws n_iters resamples in one numpy op (no Python loop)."""
    base_vals = envelope[baseline_idx]
    if base_vals.size == 0:
        return np.zeros_like(envelope)
    n = base_vals.size
    if n < 2:
        return envelope - base_vals.mean()
    idx = rng.integers(0, n, size=(n_iters, n))
    samples = base_vals[idx]
    mu = samples.mean(axis=1).mean()
    sigma = samples.std(axis=1, ddof=1).mean()
    if sigma <= 0:
        sigma = base_vals.std(ddof=1) + 1e-12
    return (envelope - mu) / sigma


def compute_subband_edges(band_spec):
    """Return list of (lo, hi) tuples splitting [low, high] into n_subbands."""
    edges = np.linspace(band_spec['low'], band_spec['high'], band_spec['n_subbands'] + 1)
    return list(zip(edges[:-1], edges[1:]))


_LOCK_COL_MAP = {
    'fixation':    'timestamps_FixationCross',
    'encoding1':   'timestamps_Encoding1',
    'encoding2':   'timestamps_Encoding2',
    'encoding3':   'timestamps_Encoding3',
    'maintenance': 'timestamps_Maintenance',
    'probe':       'timestamps_Probe',
    'response':    'timestamps_Response',
}


def collect_baseline_indices(trial_table, fs, lock_event, baseline_window, lfp_n_samples):
    col = _LOCK_COL_MAP[lock_event]
    starts = trial_table[col].to_numpy()
    starts = starts[starts > 0]
    pre, post = baseline_window
    pre_n, post_n = int(round(pre * fs)), int(round(post * fs))
    chunks = []
    for t in starts:
        center = int(round(t * fs))
        s, e = center + pre_n, center + post_n
        if s < 0 or e > lfp_n_samples or s >= e:
            continue
        chunks.append(np.arange(s, e))
    return np.concatenate(chunks) if chunks else np.array([], dtype=int)


def extract_hfa_session(lfp_clean, fs, channel_indices, trial_table, cfg, rng,
                         subband_edges):
    """Compute baseline-z-scored HFA from one session for one band.

    `subband_edges` is the list of (lo, hi) tuples for the band currently being
    processed. Caller is responsible for building it (compute_subband_edges)."""
    baseline_idx = collect_baseline_indices(
        trial_table, fs, cfg['hfa_baseline_lock_event'],
        cfg['hfa_baseline_window'], lfp_clean.shape[0]
    )
    if baseline_idx.size == 0:
        raise ValueError('empty baseline index set')
    n_chan = len(channel_indices)
    hfa = np.empty((n_chan, lfp_clean.shape[0]), dtype=np.float32)
    for ci, c in enumerate(channel_indices):
        envs = subband_envelopes(lfp_clean[:, c], fs, subband_edges, cfg['hfa_filter_order'])
        z_envs = np.empty_like(envs)
        for sb in range(envs.shape[0]):
            z_envs[sb] = bootstrap_baseline_zscore(
                envs[sb], baseline_idx, cfg['hfa_baseline_bootstrap_iters'], rng,
            )
        hfa[ci] = z_envs.mean(axis=0)
    return hfa, baseline_idx


def epoch_continuous(continuous, fs, lock_times, window):
    pre, post = window
    pre_n, post_n = int(round(pre * fs)), int(round(post * fs))
    n_t = post_n - pre_n
    T = continuous.shape[-1]
    samples, keep = [], np.ones(len(lock_times), dtype=bool)
    for ti, t in enumerate(lock_times):
        if not np.isfinite(t) or t <= 0:
            keep[ti] = False; continue
        center = int(round(t * fs)); s = center + pre_n; e = center + post_n
        if s < 0 or e > T:
            keep[ti] = False; continue
        samples.append(continuous[..., s:e])
    if not samples:
        return np.zeros((0,) + continuous.shape[:-1] + (n_t,), dtype=continuous.dtype), keep, np.array([])
    rel_time = (np.arange(n_t) + pre_n) / fs
    return np.stack(samples, axis=0), keep, rel_time


print('HFA + epoching helpers defined (multi-band)')


HFA + epoching helpers defined (multi-band)


## 3. Session Inventory

In [5]:
def hemisphere_from_location(loc):
    if not isinstance(loc, str):
        return 'unknown'
    return 'L' if loc.endswith('_left') else ('R' if loc.endswith('_right') else 'unknown')


def build_session_inventory(data_dir, regions):
    files = get_subject_files(data_dir)
    rows = []
    for f in files:
        try:
            with NWBHDF5IO(str(f), mode='r', load_namespaces=True) as io:
                nwbf = io.read()
                edf = nwbf.electrodes.to_dataframe() if nwbf.electrodes is not None else None
                trials = nwbf.trials.to_dataframe() if nwbf.trials is not None else None
                lfp_rate = float(nwbf.acquisition['LFPs'].rate) if 'LFPs' in nwbf.acquisition else np.nan
        except Exception as e:
            print(f'  [skip] {f.name}: {e}'); continue
        if edf is None:
            continue
        region_chans, _ = extract_region_channels(edf)
        row = {'subject': f.parent.name, 'session': f.stem, 'filepath': f,
               'lfp_fs': lfp_rate, 'n_trials': 0 if trials is None else len(trials)}
        for region in regions:
            row[f'n_{region.lower()}'] = len(region_chans.get(region, []))
        rows.append(row)
    return pd.DataFrame(rows)


inventory = build_session_inventory(CONFIG['data_dir'], (CONFIG['source_region'], CONFIG['target_region']))
src_col = f"n_{CONFIG['source_region'].lower()}"
tgt_col = f"n_{CONFIG['target_region'].lower()}"
thr = CONFIG['min_chans_per_region']
inventory['passes'] = (inventory[src_col] >= thr) & (inventory[tgt_col] >= thr)
passing = inventory[inventory['passes']].reset_index(drop=True)

print(f'Total sessions: {len(inventory)}')
print(f'Passing dual-coverage (>= {thr} in {CONFIG["source_region"]} & {CONFIG["target_region"]}): {len(passing)}')
print(f'Subjects passing: {passing["subject"].nunique()}')
passing[['subject', 'session', src_col, tgt_col, 'n_trials']].head(10)

Total sessions: 44
Passing dual-coverage (>= 8 in Hippocampus & Amygdala): 28
Subjects passing: 21


,subject,session,n_hippocampus,n_amygdala,n_trials
0,sub-1,sub-1_ses-1_ecephys+image,14,15,140
1,sub-1,sub-1_ses-2_ecephys+image,15,15,140
2,sub-11,sub-11_ses-1_ecephys+image,15,14,139
3,sub-12,sub-12_ses-1_ecephys+image,8,14,140
4,sub-12,sub-12_ses-2_ecephys+image,8,8,140
5,sub-13,sub-13_ses-1_ecephys+image,14,14,140
6,sub-15,sub-15_ses-1_ecephys+image,14,15,140
7,sub-16,sub-16_ses-1_ecephys+image,14,15,140
8,sub-17,sub-17_ses-1_ecephys+image,14,16,140
9,sub-19,sub-19_ses-1_ecephys+image,14,14,140


## 4. Per-Session HFA Pipeline (saves to disk)

Single function `run_session_hfa` does load → preprocess → HFA → epoch → save.
Output is one pickle per session in `HFA_CACHE_DIR` containing:
- per-region per-event tensors `(n_trials, n_chan, n_t)`
- aligned trial metadata DataFrame
- channel index → region mapping
- diagnostic info (bad channels, fs, etc.)

In [6]:
def cache_path_for_session(subject, session, band_name):
    """Per-band cache file: <HFA_CACHE_DIR>/<band_name>/hfa_<subject>_<session>.pkl."""
    cache_dir = CONFIG['hfa_cache_dir'] / band_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f'hfa_{subject}_{session}.pkl'


def run_session_hfa_all_bands(filepath, cfg, rng=None, force=False, verbose=False):
    """Load NWB once, preprocess LFP once, then compute and cache HFA for every
    band in cfg['bands'] in turn. One pickle per (session, band).

    Returns a small summary dict for logging; the heavy data lives on disk."""
    rng = rng if rng is not None else np.random.default_rng(cfg['random_seed'])
    subject = filepath.parent.name
    session = filepath.stem

    bands = cfg['bands']
    if not force:
        if all(cache_path_for_session(subject, session, b).exists() for b in bands):
            return {'subject': subject, 'session': session,
                    'bands_done': [], 'bands_skipped': list(bands),
                    'note': 'all bands cached'}

    nwb = load_nwb_file(filepath)
    try:
        clean_lfp, fs, edf, prep_diag = preprocess_session_lfp(nwb, cfg, verbose=verbose)
        trials = nwb['trials']
        if cfg['use_only_correct'] and 'response_accuracy' in trials.columns:
            trials = trials[trials['response_accuracy'] == 1].copy()

        region_chans_local, _ = extract_region_channels(edf)
        n_chan_lfp = clean_lfp.shape[1]
        bad_set = set(prep_diag['bad_idx'])

        src = [c for c in region_chans_local.get(cfg['source_region'], [])
               if c < n_chan_lfp and c not in bad_set]
        tgt = [c for c in region_chans_local.get(cfg['target_region'], [])
               if c < n_chan_lfp and c not in bad_set]
        if len(src) == 0 or len(tgt) == 0:
            return None
        all_chans = src + tgt
        n_src = len(src)

        bands_done, bands_skipped = [], []
        for band_name, band_spec in bands.items():
            cache_path = cache_path_for_session(subject, session, band_name)
            if cache_path.exists() and not force:
                bands_skipped.append(band_name)
                continue

            subband_edges = compute_subband_edges(band_spec)
            hfa, _ = extract_hfa_session(
                clean_lfp, fs, all_chans, trials, cfg, rng,
                subband_edges=subband_edges,
            )

            event_tensors = {}
            for event_name, window in cfg['epoch_windows'].items():
                lock_col = _LOCK_COL_MAP[event_name]
                lock_times = trials[lock_col].to_numpy()
                full_tensor, keep, rel_time = epoch_continuous(hfa, fs, lock_times, window)
                if full_tensor.shape[0] == 0:
                    event_tensors[event_name] = {
                        'src': np.zeros((0, n_src, 0)),
                        'tgt': np.zeros((0, len(tgt), 0)),
                        'rel_time': np.array([]),
                        'meta': trials.iloc[:0].copy(),
                    }
                    continue
                event_tensors[event_name] = {
                    'src': full_tensor[:, :n_src, :],
                    'tgt': full_tensor[:, n_src:, :],
                    'rel_time': rel_time,
                    'meta': trials.loc[keep].reset_index(drop=True),
                }

            result = {
                'subject': subject, 'session': session, 'fs': fs,
                'src_chans': src, 'tgt_chans': tgt,
                'src_region': cfg['source_region'], 'tgt_region': cfg['target_region'],
                'event_tensors': event_tensors,
                'bad_chans': prep_diag['bad_idx'],
                'flag_cols': prep_diag['flag_cols'],
                'n_chan_lfp': n_chan_lfp,
                'band_name': band_name,
                'band_spec': band_spec,
                'subband_edges': subband_edges,
            }
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
            bands_done.append(band_name)

        return {'subject': subject, 'session': session,
                'src_chans': src, 'tgt_chans': tgt,
                'bad_chans': prep_diag['bad_idx'],
                'bands_done': bands_done, 'bands_skipped': bands_skipped}
    finally:
        nwb['io'].close()


print('per-session multi-band HFA pipeline defined')


per-session multi-band HFA pipeline defined


## 5. All-Sessions HFA Sweep (threaded)

Extracts HFA + builds per-event tensors for every passing session and writes
each one to a pickle in `HFA_CACHE_DIR`. Re-runs are cached: deleting a pickle
forces re-extraction for that session only.

In [7]:
FORCE_RECOMPUTE_HFA = False    # set True to ignore cache and rebuild every (session, band)

def _hfa_one(sess_i, sess):
    try:
        result = run_session_hfa_all_bands(
            sess['filepath'], CONFIG,
            rng=np.random.default_rng(CONFIG['random_seed'] + sess_i),
            force=FORCE_RECOMPUTE_HFA,
        )
        return ('ok', sess_i, sess, result)
    except Exception as e:
        return ('err', sess_i, sess, repr(e))


t0 = time.time()
all_session_summaries = []
skipped_hfa = []
sessions_list = list(passing.iterrows())
print(f'HFA sweep (multi-band): {len(sessions_list)} sessions, '
      f'{len(CONFIG["bands"])} bands, {CONFIG["n_workers"]} threads')
with ThreadPoolExecutor(max_workers=CONFIG['n_workers']) as exe:
    futs = [exe.submit(_hfa_one, i, sess) for i, sess in sessions_list]
    for fut in as_completed(futs):
        status, sess_i, sess, payload = fut.result()
        tag = f"{sess['subject']}/{sess['session']}"
        if status == 'ok' and payload is not None:
            all_session_summaries.append(payload)
            done = ','.join(payload.get('bands_done', [])) or '\u2014'
            skipd = ','.join(payload.get('bands_skipped', [])) or '\u2014'
            print(f'  [done] {tag}  '
                  f"src={len(payload.get('src_chans', []))}  "
                  f"tgt={len(payload.get('tgt_chans', []))}  "
                  f'bands_new=[{done}] bands_cached=[{skipd}]')
        else:
            detail = payload if status == 'err' else 'no plottable channels'
            print(f'  [skip] {tag}: {detail}')
            skipped_hfa.append((sess['subject'], sess['session']))

print(f'\nDone: {len(all_session_summaries)} sessions OK, '
      f'{len(skipped_hfa)} skipped in {time.time()-t0:.0f}s')
print(f'Cache layout: {HFA_CACHE_DIR}/<band_name>/hfa_<subject>_<session>.pkl')


HFA sweep (multi-band): 28 sessions, 5 bands, 4 threads
  [done] sub-12/sub-12_ses-1_ecephys+image  src=8  tgt=14  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-12/sub-12_ses-2_ecephys+image  src=8  tgt=8  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-11/sub-11_ses-1_ecephys+image  src=15  tgt=14  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-1/sub-1_ses-1_ecephys+image  src=14  tgt=15  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-1/sub-1_ses-2_ecephys+image  src=15  tgt=15  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-16/sub-16_ses-1_ecephys+image  src=14  tgt=15  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-13/sub-13_ses-1_ecephys+image  src=14  tgt=14  bands_new=[theta,alpha,beta,low_gamma,high_gamma] bands_cached=[—]
  [done] sub-15/sub-15_ses-1_ecephys+image  src=14  tgt=15  ba